# Data and features — Member 1

- Prepare **143 features** for authorship attribution across 20 authors.
- Load data → validate → extract features → save shared files.
- All code is included here; run cells from top to bottom.
- No classifier training yet. Test data stays excluded by default.

## 1. Setup

In [15]:
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
import csv
import hashlib
import json
import platform
import re
import string

import nltk
import numpy as np
from nltk.tag import PerceptronTagger
from nltk.tokenize import TreebankWordTokenizer

def find_project_root(start):
    """Locate the folder holding both this notebook and the supplied data."""
    start = Path(start).resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / "TxMM_assignment3",
                          parent / "3" / "TxMM_assignment3"):
            if ((candidate / "data_features.ipynb").is_file()
                    and (candidate / "A3data_2627_pan2020").is_dir()):
                return candidate
    raise FileNotFoundError(
        "Cannot locate TxMM_assignment3. Start Jupyter in the project or its parent folder."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "A3data_2627_pan2020"
OUTPUT_DIR = PROJECT_ROOT / "output" / "features"
NLTK_DATA_DIR = PROJECT_ROOT / ".cache" / "nltk_data"
DOWNLOAD_NLTK = True  
INCLUDE_TEST = False 
OVERWRITE_ARTIFACTS = True

if str(NLTK_DATA_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DATA_DIR))
try:
    nltk.data.find("taggers/averaged_perceptron_tagger_eng/")
except LookupError:
    if not DOWNLOAD_NLTK:
        raise RuntimeError("English POS model missing. Set DOWNLOAD_NLTK = True and rerun this cell.")
    NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not nltk.download("averaged_perceptron_tagger_eng",
                         download_dir=str(NLTK_DATA_DIR), raise_on_error=True):
        raise RuntimeError("English POS model download failed.")

print("Project:", PROJECT_ROOT.name)
print("Data:", DATA_DIR.relative_to(PROJECT_ROOT))
print("Shared artifacts:", OUTPUT_DIR.relative_to(PROJECT_ROOT))
print({"python": platform.python_version(), "numpy": np.__version__, "nltk": nltk.__version__})

Project: TxMM_assignment3
Data: A3data_2627_pan2020
Shared artifacts: output\features
{'python': '3.13.9', 'numpy': '2.3.5', 'nltk': '3.9.2'}


## 2. Load and validate

- Columns: unnamed snippet ID, `text`, `author`.


In [3]:
@dataclass(frozen=True)
class Dataset:
    ids: tuple
    texts: tuple
    authors: tuple

    def __len__(self):
        return len(self.ids)

    def summary(self):
        return {"rows": len(self), "authors": dict(sorted(Counter(self.authors).items()))}


def load_data(path):
    """Read BOM-aware CSV; reject malformed rows, empty fields and duplicate IDs."""
    path = Path(path)
    ids, texts, authors = [], [], []
    seen = set()
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["", "text", "author"]:
            raise ValueError(f"{path}: expected columns ['', 'text', 'author']; got {reader.fieldnames}")
        for row in reader:
            location = f"{path}: CSV line {reader.line_num}"
            if None in row or any(value is None for value in row.values()):
                raise ValueError(f"{location}: wrong number of fields")
            if any(not value.strip() for value in row.values()):
                raise ValueError(f"{location}: empty ID, text or author")
            snippet_id, author = row[""].strip(), row["author"].strip()
            if snippet_id in seen:
                raise ValueError(f"{location}: duplicate snippet ID {snippet_id!r}")
            seen.add(snippet_id)
            ids.append(snippet_id)
            texts.append(row["text"])
            authors.append(author)
    if not ids:
        raise ValueError(f"{path}: dataset contains no rows")
    return Dataset(tuple(ids), tuple(texts), tuple(authors))

### Split checks

- Train: fit models. Dev: tune choices. Test: final evaluation only.


In [4]:
def validate_splits(splits):
    """Reject overlapping snippet IDs and labels absent from the training split."""
    if "train" not in splits:
        raise ValueError("A training split is required")
    training_authors = set(splits["train"].authors)
    seen = set()
    for name, dataset in splits.items():
        overlap = seen.intersection(dataset.ids)
        if overlap:
            raise ValueError(f"{name}: snippet IDs overlap other splits: {sorted(overlap)[:5]}")
        unseen = set(dataset.authors) - training_authors
        if unseen:
            raise ValueError(f"{name}: authors missing from training: {sorted(unseen)}")
        seen.update(dataset.ids)

In [5]:
split_names = ["train", "dev"] + (["test"] if INCLUDE_TEST else [])
source_paths = {name: DATA_DIR / f"pan2627_{name}_data.csv" for name in split_names}
datasets = {name: load_data(path) for name, path in source_paths.items()}
validate_splits(datasets)

print(f"{'Split':<10} {'Rows':>7} {'Authors':>9} {'Min/author':>12} {'Max/author':>12}")
for name, dataset in datasets.items():
    counts = Counter(dataset.authors)
    print(f"{name:<10} {len(dataset):>7} {len(counts):>9} {min(counts.values()):>12} {max(counts.values()):>12}")

Split         Rows   Authors   Min/author   Max/author
train         2138        20           51          150
dev            268        20            5           21


- Author counts are unequal; no resampling in this phase.
- Suggested metric: **macro-F1**, giving each author equal weight.


## 3. Feature groups

- **Function words (56):** 
- **Character frequencies (26):** 
- **Idiosyncratic (12):** 
- **Authorship stylistic (12):** 
- **POS tags (37):** 3
- Total: **143 features**; the assignment requires at least 50, including lexical and syntactic features.


In [6]:
FUNCTION_WORDS = tuple("a an the and or but if while because as until of at by for with about against between into through during before after above below to from up down in out on off over under again further then once here there when where why how all any both each few more most other some such".split())
POS_TAGS = tuple("CC CD DT EX FW IN JJ JJR JJS LS MD NN NNS NNP NNPS PDT POS PRP PRP$ RB RBR RBS RP SYM TO UH VB VBD VBG VBN VBP VBZ WDT WP WP$ WRB".split())
PUNCTUATION = {"period": ".", "comma": ",", "semicolon": ";", "colon": ":", "question": "?", "exclamation": "!", "apostrophe": "'", "quote": '\"', "hyphen": "-", "open_parenthesis": "(", "close_parenthesis": ")", "ellipsis": "…"}
STYLE_NAMES = ("mean_word_length", "std_word_length", "type_token_ratio", "hapax_ratio", "mean_sentence_words", "std_sentence_words", "short_word_ratio", "long_word_ratio", "uppercase_char_ratio", "digit_char_ratio", "whitespace_char_ratio", "newline_char_ratio")
GROUP_DESCRIPTIONS = {
    "function_words": "Fixed function-word counts divided by lexical word count; captures habitual word choice.",
    "character_frequencies": "Lowercase a-z counts divided by total character count; captures spelling and character preferences.",
    "idiosyncratic": "Our operational definition of Idiosyncratic: selected punctuation counts divided by total character count; not a reproduction of the paper’s complete feature group.",
    "authorship_stylistic": "Word/sentence length and vocabulary diversity plus character-class ratios; captures broad writing style.",
    "pos_tags": "Penn Treebank tag frequencies divided by all Treebank tokens, plus OTHER; captures grammatical patterns.",
}
WORD_RE = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*", re.UNICODE)


GROUP_LABELS = {
    "function_words": "Function words",
    "character_frequencies": "Character frequencies",
    "idiosyncratic": "Idiosyncratic",
    "authorship_stylistic": "Authorship stylistic",
    "pos_tags": "POS tags",
}

### Preprocessing choices


In [7]:
class FeatureExtractor:

    def __init__(self, nltk_data_dir=None):
        if nltk_data_dir is not None:
            directory = str(Path(nltk_data_dir).resolve())
            if directory not in nltk.data.path:
                nltk.data.path.insert(0, directory)
        try:
            self.tagger = PerceptronTagger(lang="eng")
        except LookupError as exc:
            raise RuntimeError("Missing NLTK English POS model. Set DOWNLOAD_NLTK = True and rerun the setup cell (requires network access).") from exc
        self.tokenizer = TreebankWordTokenizer()
        groups = {
            "function_words": [f"function_words_{word}" for word in FUNCTION_WORDS],
            "character_frequencies": [f"character_frequencies_{char}" for char in string.ascii_lowercase],
            "idiosyncratic": [f"idiosyncratic_{name}" for name in PUNCTUATION],
            "authorship_stylistic": [f"authorship_stylistic_{name}" for name in STYLE_NAMES],
            "pos_tags": [f"pos_tags_{tag}" for tag in (*POS_TAGS, "OTHER")],
        }
        self.feature_names = []
        self.feature_groups = {}
        for group, names in groups.items():
            start = len(self.feature_names)
            self.feature_names.extend(names)
            self.feature_groups[group] = list(range(start, len(self.feature_names)))

    def transform(self, texts):
        """Return finite float64 rows in the supplied order, including empty input."""
        rows = [self._extract(text) for text in texts]
        matrix = np.asarray(rows, dtype=np.float64).reshape((-1, len(self.feature_names)))
        if not np.isfinite(matrix).all():
            raise ValueError("Feature extraction produced non-finite values")
        return matrix

    def _extract(self, text):
        if not isinstance(text, str):
            raise TypeError("Every text must be a string")
        words = WORD_RE.findall(text.lower())
        word_counts = Counter(words)
        n_words, n_chars = max(len(words), 1), max(len(text), 1)
        char_counts = Counter(text.lower())
        values = [word_counts[word] / n_words for word in FUNCTION_WORDS]
        values += [char_counts[char] / n_chars for char in string.ascii_lowercase]
        values += [text.count(char) / n_chars for char in PUNCTUATION.values()]
        lengths = [len(word) for word in words]
        sentence_lengths = [len(WORD_RE.findall(sentence)) for sentence in re.split(r"[.!?]+", text)]
        sentence_lengths = [length for length in sentence_lengths if length]
        values += [
            float(np.mean(lengths)) if lengths else 0.0,
            float(np.std(lengths)) if lengths else 0.0,
            len(word_counts) / n_words,
            sum(count == 1 for count in word_counts.values()) / n_words,
            float(np.mean(sentence_lengths)) if sentence_lengths else 0.0,
            float(np.std(sentence_lengths)) if sentence_lengths else 0.0,
            sum(length <= 3 for length in lengths) / n_words,
            sum(length >= 7 for length in lengths) / n_words,
            sum(char.isupper() for char in text) / n_chars,
            sum(char.isdigit() for char in text) / n_chars,
            sum(char.isspace() for char in text) / n_chars,
            text.count("\n") / n_chars,
        ]
        tokens = self.tokenizer.tokenize(text)
        tags = Counter(tag for _, tag in self.tagger.tag(tokens))
        denominator = max(len(tokens), 1)
        values += [tags[tag] / denominator for tag in POS_TAGS]
        values += [sum(count for tag, count in tags.items() if tag not in POS_TAGS) / denominator]
        return values

    def metadata(self):
        return {"schema_version": 2, "feature_names": self.feature_names,
                "feature_groups": self.feature_groups, "group_descriptions": GROUP_DESCRIPTIONS,
                "feature_group_labels": GROUP_LABELS,
                "feature_count": len(self.feature_names), "learned_from_assignment_data": False,
                "pos_model": "NLTK averaged_perceptron_tagger_eng",
                "sentence_boundaries": "Runs of .!?; nonempty lexical segments only",
                "word_definition": "Unicode letters with optional internal apostrophes; lowercase"}

In [8]:
extractor = FeatureExtractor(NLTK_DATA_DIR)
for group, indices in extractor.feature_groups.items():
    print(f"{group:<16} {len(indices):>3} columns")
print("Total:", len(extractor.feature_names))

all_indices = [i for indices in extractor.feature_groups.values() for i in indices]
assert all_indices == list(range(len(extractor.feature_names)))
assert len(set(extractor.feature_names)) == len(extractor.feature_names)
assert len(extractor.feature_names) >= 50

for group, indices in extractor.feature_groups.items():
    print("\n" + GROUP_LABELS[group] + ":")
    print(", ".join(extractor.feature_names[i] for i in indices))

function_words    56 columns
character_frequencies  26 columns
idiosyncratic     12 columns
authorship_stylistic  12 columns
pos_tags          37 columns
Total: 143

Function words:
function_words_a, function_words_an, function_words_the, function_words_and, function_words_or, function_words_but, function_words_if, function_words_while, function_words_because, function_words_as, function_words_until, function_words_of, function_words_at, function_words_by, function_words_for, function_words_with, function_words_about, function_words_against, function_words_between, function_words_into, function_words_through, function_words_during, function_words_before, function_words_after, function_words_above, function_words_below, function_words_to, function_words_from, function_words_up, function_words_down, function_words_in, function_words_out, function_words_on, function_words_off, function_words_over, function_words_under, function_words_again, function_words_further, function_words_then, fun

## 4. Build feature matrices

- Extract from text only; keep IDs and labels alongside each row.
- Use identical columns and ordering for every split.
- No feature choices learned from dev/test data.
- Runtime depends on the machine; allow about a minute.

In [9]:
matrices = {}
for name, dataset in datasets.items():
    matrices[name] = extractor.transform(dataset.texts)
    assert matrices[name].shape == (len(dataset), len(extractor.feature_names))
    assert np.isfinite(matrices[name]).all()
    print(f"{name}: {matrices[name].shape[0]} rows × {matrices[name].shape[1]} features")

train: 2138 rows × 143 features
dev: 268 rows × 143 features


## 5. Save shared output

- Location: `output/features/`.
- `train.npz`, `dev.npz`: `X` = features, `ids` = snippet IDs, `y` = author labels.
- `feature_metadata.json`: names, groups, definitions, versions, counts and source hashes.

In [16]:
metadata = extractor.metadata()
metadata["versions"] = {"python": platform.python_version(), "numpy": np.__version__, "nltk": nltk.__version__}
metadata["splits"] = {
    name: {**dataset.summary(), "source_file": source_paths[name].name,
           "sha256": hashlib.sha256(source_paths[name].read_bytes()).hexdigest()}
    for name, dataset in datasets.items()
}
payloads = {
    name: {"X": matrices[name], "ids": np.asarray(dataset.ids, dtype=str),
           "y": np.asarray(dataset.authors, dtype=str)}
    for name, dataset in datasets.items()
}
if not INCLUDE_TEST and (OUTPUT_DIR / "test.npz").exists():
    raise ValueError("Use a fresh OUTPUT_DIR, or rebuild all splits after freezing the approach.")

# Preflight all existing outputs before writing any files.
for name, payload in payloads.items():
    target = OUTPUT_DIR / f"{name}.npz"
    if target.exists() and not OVERWRITE_ARTIFACTS:
        with np.load(target, allow_pickle=False) as saved:
            identical = set(saved.files) == set(payload) and all(
                np.array_equal(saved[key], value) for key, value in payload.items())
        if not identical:
            raise ValueError(f"{target.name} differs; choose another output directory or enable replacement.")
metadata_path = OUTPUT_DIR / "feature_metadata.json"
if metadata_path.exists() and not OVERWRITE_ARTIFACTS:
    if json.loads(metadata_path.read_text(encoding="utf-8")) != metadata:
        raise ValueError("Metadata differs; choose another output directory or enable replacement.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, payload in payloads.items():
    target = OUTPUT_DIR / f"{name}.npz"
    if not target.exists() or OVERWRITE_ARTIFACTS:
        np.savez_compressed(target, **payload)
if not metadata_path.exists() or OVERWRITE_ARTIFACTS:
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print("Artifacts saved or verified identical:", ", ".join(f"{name}.npz" for name in payloads))
print("Metadata: feature_metadata.json")
print("Saved in:", OUTPUT_DIR.relative_to(PROJECT_ROOT))

Artifacts saved or verified identical: train.npz, dev.npz
Metadata: feature_metadata.json
Saved in: output\features


In [28]:
for name, dataset in datasets.items():
    with np.load(OUTPUT_DIR / f"{name}.npz", allow_pickle=False) as saved:
        np.testing.assert_array_equal(saved["X"], matrices[name])
        assert saved["ids"].tolist() == list(dataset.ids)
        assert saved["y"].tolist() == list(dataset.authors)
        assert np.isfinite(saved["X"]).all()
        if name == "train":
            train = dataset
        elif name == "dev":
            dev = dataset
print("Round-trip checks passed: values, row order and labels are preserved.")

Round-trip checks passed: values, row order and labels are preserved.


# Training the classifier

In [41]:
from src.model import Classifier

feature_names = extractor.metadata()['feature_names']

cls = Classifier(train, dev, feature_names)
cls.train()

AttributeError: 'Dataset' object has no attribute 'x'